# CD Fibroblasts: Pseudotime and Marker Gene Expression

Analyses on connective tissue cells from the CD atlas:

1. **Palantir pseudotime** from ADAMDEC1+ Fib root → Crypt Top Fib
   and Myofibroblast terminals.
2. **DPT validation** using scanpy diffusion pseudotime with PAGA.
3. **Marker gene dotplots**: pro-inflammatory and pro-fibrotic signatures.

Pericytes are excluded to focus on the stromal fibroblast compartment.

**Inputs** (exported from `03_cd_fibroblast_subclustering.R`):
- `connective/con_iter2_umap.csv`      – UMAP coordinates
- `connective/con_harmony_iter2.csv`   – Harmony embedding (20 dims)
- `merged_cd/cleaned_annoed_all_cell_types.h5ad` – full annotated atlas

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
CON_DIR    = f"{DATA_DIR}/connective"
MERGED_DIR = f"{DATA_DIR}/merged_cd"
OUTPUT_DIR = f"{DATA_DIR}/connective"

## 1. Subset fibroblasts and attach embeddings

In [ ]:
# Load full atlas and subset connective tissue cells
scrna = sc.read_h5ad(f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")
fib   = scrna[scrna.obs['cell category'] == 'Connective'].copy()
print(fib.obs['label'].value_counts())

In [ ]:
# Attach fibroblast UMAP coordinates (exported from R)
umap_fib = pd.read_csv(f"{CON_DIR}/con_iter2_umap.csv")
fib.obsm['X_umap'] = umap_fib.set_index('cell')[['umapharmonyconiter2_1',
                                                   'umapharmonyconiter2_2']].loc[fib.obs_names].to_numpy()

# Attach Harmony embedding
harm = pd.read_csv(f"{CON_DIR}/con_harmony_iter2.csv", index_col=0)
fib.obsm['X_harmony'] = harm.loc[fib.obs_names].to_numpy()

In [ ]:
# Remove pericytes – focus on stromal fibroblast compartment
exclude_peri = ['CD36+ Pericyte', 'RERGL+ Contractile Pericyte']
fib = fib[~fib.obs['label'].isin(exclude_peri)].copy()

sc.pl.umap(fib, color='label', frameon=False, s=5)

## 2. Palantir pseudotime

Root: ADAMDEC1+ Fib centroid (most naïve stromal state).

In [ ]:
# Diffusion maps and multi-scale space
dm_res    = palantir.utils.run_diffusion_maps(fib, n_components=5, pca_key='X_harmony')
ms_data   = palantir.utils.determine_multiscale_space(fib)
imputed_X = palantir.utils.run_magic_imputation(fib)

palantir.plot.plot_diffusion_components(fib)
plt.show()

In [ ]:
# Find root cell: closest to ADAMDEC1+ Fib UMAP centroid
coords     = fib.obsm['X_umap']
early_mask = fib.obs['label'] == 'ADAMDEC1+ Fib'
early_idx  = np.where(early_mask)[0]
centroid   = coords[early_idx].mean(axis=0)
root_cell  = fib.obs_names[early_idx[np.argmin(
    ((coords[early_idx] - centroid) ** 2).sum(axis=1)
)]]
print(f"Root cell: {root_cell}")

start_states = pd.Series(['ADAMDEC1+ Fib (root)'], index=[root_cell])
palantir.plot.highlight_cells_on_umap(fib, start_states, s=5)
plt.show()

In [ ]:
# Run Palantir from ADAMDEC1+ Fib root
pr_res = palantir.core.run_palantir(fib, root_cell, num_waypoints=500)

palantir.plot.plot_palantir_results(fib, s=3)
plt.show()

## 3. DPT validation (diffusion pseudotime with PAGA)

In [ ]:
fib_cp = fib.copy()

sc.pp.neighbors(fib_cp, use_rep='X_harmony', n_neighbors=20, n_pcs=20)
sc.tl.paga(fib_cp, groups='label')
sc.pl.paga(fib_cp, plot=False)
sc.tl.umap(fib_cp, init_pos='paga')

# DPT from ADAMDEC1+ Fib root
fib_cp.uns['iroot'] = np.flatnonzero(fib_cp.obs['label'] == 'ADAMDEC1+ Fib')[0]
sc.tl.dpt(fib_cp)

# Restore original UMAP
fib_cp.obsm['X_umap'] = fib.obsm['X_umap']

In [ ]:
umap_purple = LinearSegmentedColormap.from_list(
    'umap_purple', ['#e8d4cf', '#a16f8d', '#2d223a']
)
ax = sc.pl.umap(
    fib_cp, color='dpt_pseudotime',
    cmap=umap_purple, vmin=0, vmax='p99',
    na_color='#e8d4cf', frameon=False, show=False
)
ax.set_title('Diffusion Pseudotime (DPT validation)', fontsize=14)
plt.show()

## 4. Marker gene dotplots

Pro-inflammatory and pro-fibrotic gene signatures across CD fibroblast subtypes.

In [ ]:
# Ordered fibroblast subtypes (ADAMDEC1+ Fib as anchor)
ct_order = [
    'ADAMDEC1+ Fib', 'OGN+RSPO3+ Fib', 'T cell-Interacting Fib',
    'Activated Fib', 'ADAMDEC1+ Activated Fib',
    'VSTM2A+ Crypt Top Fib', 'PLA2G2A+ ECM Fib',
    'HHIP+ Myofibroblast', 'GREM2+ Myofibroblast'
]

# Subset to non-cycling, non-SELENOP+ fibroblasts
fib_cl = fib[~fib.obs['label'].isin(['SELENOP+ Fib'])].copy()
fib_cl.obs['label'] = pd.Categorical(fib_cl.obs['label'], categories=ct_order)

In [ ]:
pro_inf_genes = ['IL6', 'CXCL1', 'CXCL2', 'CXCL3', 'CXCL5', 'CCL2',
                 'NFKBIA', 'TNFAIP3', 'ICAM1', 'PTGS2', 'MMP3']
profib_genes  = ['TGFB1', 'LGALS3', 'COL1A1', 'COL1A2',
                 'FN1', 'TIMP1', 'SERPINE1', 'INHBA']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.dotplot(
    fib_cl, var_names=pro_inf_genes, groupby='label',
    ax=axes[0], show=False, standard_scale=None, vmin=0, vmax=50
)
axes[0].set_title('Pro-inflammatory genes', fontsize=14, fontweight='bold')

sc.pl.dotplot(
    fib_cl, var_names=profib_genes, groupby='label',
    ax=axes[1], show=False, standard_scale=None, vmin=0, vmax=50
)
axes[1].set_title('Pro-fibrotic genes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/cd_fibroblast_marker_dotplots.pdf", bbox_inches='tight', dpi=300)
plt.show()